# A/B-тест рекламной кампании: есть ли инкрементальный эффект

**Вопрос бизнеса:** приносит ли реклама дополнительные конверсии, или они случились бы и без неё? Стоит ли её продолжать?

## 1. Дизайн эксперимента

**Дизайн:**

1. Гипотеза: показ рекламы повышает конверсию в сравнении с социальной рекламой на том же месте, потому что реклама убеждает часть людей, которые не совершили бы без нее целевое действие
2. Метрика: конверсия, то есть число пользователей, для которых `converted = True` / число всех пользователей в группе. Считается отдельно для `ad` и `psa`.
3. Единица рандомизации: пользователи. Единица анализа — также пользователи.
4. Зачем PSA: в данном случае контрольная группа попадает на те же рекламные места, но отличается лишь их содержание. Если бы контрольной группе заменили рекламу на пустое место, то пришлось бы также сравнивать эффект самого факта показа для обеих групп и эффект нельзя было бы приписать рекламе.
5. Почему маленький контроль: каждый человек в контроле — потенциально упущенная прибыль, поэтому ищем компромисс между риском для бюджета и точностью.

## 2. Загрузка и проверка данных

In [1]:
import duckdb
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

df = pd.read_csv("../data/d2/marketing_AB.csv")
df = df.drop(columns="Unnamed: 0")
df.columns = ["user_id", "test_group", "converted", "total_ads", "most_ads_day", "most_ads_hour"]
df.head()

,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14


In [2]:
duckdb.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT user_id) AS n_users
    FROM df
""").df()

,n_rows,n_users
0,588101,588101


In [3]:
duckdb.sql("""
    SELECT COUNT(*) - COUNT(user_id)       AS user_id_nulls,
           COUNT(*) - COUNT(test_group)    AS test_group_nulls,
           COUNT(*) - COUNT(converted)     AS converted_nulls,
           COUNT(*) - COUNT(total_ads)     AS total_ads_nulls,
           COUNT(*) - COUNT(most_ads_day)  AS most_ads_day_nulls,
           COUNT(*) - COUNT(most_ads_hour) AS most_ads_hour_nulls
    FROM df
""").df()

,user_id_nulls,test_group_nulls,converted_nulls,total_ads_nulls,most_ads_day_nulls,most_ads_hour_nulls
0,0,0,0,0,0,0


In [4]:
display(duckdb.sql("""
    SELECT DISTINCT test_group
    FROM df
""").df())

duckdb.sql("""
    SELECT COUNT(*) AS users_in_both_groups
    FROM (
        SELECT user_id
        FROM df
        GROUP BY user_id
        HAVING COUNT(DISTINCT test_group) > 1
    ) AS t
""").df()

,test_group
0,psa
1,ad


,users_in_both_groups
0,0


In [5]:
duckdb.sql("""
    SELECT MIN(total_ads) AS min_ads,
           MAX(total_ads) AS max_ads,
           QUANTILE_CONT(total_ads, 0.99) AS p99_ads,
           MIN(most_ads_hour) AS min_hour,
           MAX(most_ads_hour) AS max_hour
    FROM df
""").df()

,min_ads,max_ads,p99_ads,min_hour,max_hour
0,1,2065,202.0,0,23


**Вывод по качеству данных:** 588 101 строка и столько же уникальных `user_id`, то есть один пользователь — одна строка. Пропусков нет ни в одной колонке. Групп две (`ad`, `psa`), пользователей сразу в двух группах нет. `most_ads_hour` лежит в 0-23, `total_ads` от 1 до 2065. Максимум выглядит подозрительно при 99-м перцентиле 202 (возможны боты или очень активные пользователи), но `total_ads` не входит в метрику, поэтому строки можно не удалять, чистка по признаку, измеренному после начала показа, могла бы сместить сравнение.

In [6]:
display(duckdb.sql("""
    SELECT test_group,
           COUNT(*) AS users,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS share_pct,
           MEDIAN(total_ads) AS median_ads,
           ROUND(AVG(total_ads), 2) AS avg_ads
    FROM df
    GROUP BY test_group
    ORDER BY test_group
""").df())

duckdb.sql("""
    SELECT most_ads_day,
           ROUND(100.0 * COUNT(*) FILTER (WHERE test_group = 'ad')
                 / (SELECT COUNT(*) FROM df WHERE test_group = 'ad'), 2) AS ad_pct,
           ROUND(100.0 * COUNT(*) FILTER (WHERE test_group = 'psa')
                 / (SELECT COUNT(*) FROM df WHERE test_group = 'psa'), 2) AS psa_pct
    FROM df
    GROUP BY most_ads_day
    ORDER BY most_ads_day
""").df()

,test_group,users,share_pct,median_ads,avg_ads
0,ad,564577,96.0,13.0,24.82
1,psa,23524,4.0,12.0,24.76


,most_ads_day,ad_pct,psa_pct
0,Friday,15.73,16.17
1,Monday,14.80,14.89
2,Saturday,13.96,12.15
3,Sunday,14.58,13.00
4,Thursday,14.01,16.60
5,Tuesday,13.21,12.36
6,Wednesday,13.71,14.84


**Группы и сопоставимость**

`ad` — 564 577 человек (96 %), `psa` — 23 524 (4 %). Перекос задуман: контроль сделали маленьким, чтобы не терять конверсии.

- *План разбиения неизвестен.* В описании датасета не сказано, в какой пропорции людей хотели разделить, поэтому проверить, что сплит соблюдён, нельзя. Это нормально для открытых данных.
- *Состояния «до» нет.* В данных нет ни одного признака, измеренного до старта кампании, поэтому проверить, что группы были одинаковыми в начале, нельзя. Приходится опираться на то, что разбиение было случайным.
- *Косвенные признаки совпадают.* Среднее число показов на человека почти одинаковое (24,82 и 24,76). По дням недели доли отличаются до 2,6 п.п. (четверг: 14,0 % и 16,6 %), но это день с максимумом показов, он измеряется во время кампании и на сравнение конверсий не влияет.

## 3. Анализ

In [7]:
groups = duckdb.sql("""
    SELECT test_group,
           COUNT(*) AS users,
           COUNT(*) FILTER (WHERE converted) AS conversions,
           ROUND(100.0 * COUNT(*) FILTER (WHERE converted) / COUNT(*), 3) AS cr_pct
    FROM df
    GROUP BY test_group
    ORDER BY test_group
""").df()
groups

,test_group,users,conversions,cr_pct
0,ad,564577,14423,2.555
1,psa,23524,420,1.785


In [8]:
ad = groups[groups["test_group"] == "ad"].iloc[0]
psa = groups[groups["test_group"] == "psa"].iloc[0]
count_ad, n_ad = int(ad["conversions"]), int(ad["users"])
count_psa, n_psa = int(psa["conversions"]), int(psa["users"])

cr_ad, cr_psa = count_ad / n_ad, count_psa / n_psa
diff = cr_ad - cr_psa

print(f"абсолютная разница: {diff * 100:.3f} п.п.")
print(f"относительная разница: {(cr_ad / cr_psa - 1) * 100:.1f} %")

абсолютная разница: 0.769 п.п.
относительная разница: 43.1 %


In [9]:
z, p = proportions_ztest([count_ad, count_psa], [n_ad, n_psa])
print(f"z = {z:.2f}, p = {p:.2e}")

z = 7.37, p = 1.71e-13


In [10]:
low, high = confint_proportions_2indep(count_ad, n_ad, count_psa, n_psa, method="wald", compare="diff")
rlow, rhigh = confint_proportions_2indep(count_ad, n_ad, count_psa, n_psa, method="log", compare="ratio")

print(f"разница: {diff * 100:.2f} п.п., 95% CI [{low * 100:.2f}; {high * 100:.2f}] п.п.")
print(f"относительная разница, 95% CI [{(rlow - 1) * 100:.1f}; {(rhigh - 1) * 100:.1f}] %")
print(f"дополнительных конверсий на 1000 пользователей: {diff * 1000:.1f}, 95% CI [{low * 1000:.1f}; {high * 1000:.1f}]")

разница: 0.77 п.п., 95% CI [0.60; 0.94] п.п.
относительная разница, 95% CI [30.0; 57.5] %
дополнительных конверсий на 1000 пользователей: 7.7, 95% CI [6.0; 9.4]


**Что получилось:** конверсия `ad` — 2,555 % (14 423 из 564 577), `psa` — 1,785 % (420 из 23 524). Разница +0,77 п.п., 95 % CI [0,60; 0,94], относительная +43,1 %, 95 % CI [30,0; 57,5]. Это 7,7 дополнительной конверсии на 1000 пользователей, 95 % CI [6,0; 9,4]. z = 7,37, p ≈ 1,7·10⁻¹³: если бы реклама не меняла конверсию, разница такого размера или больше возникала бы с такой вероятностью. $H_0$ отвергаем на уровне 5 %, интервал не содержит 0.

## 4. Вывод продакту

**Вывод продакту:**

1. Проверяли, даёт ли реклама дополнительные конверсии: сравнили 564 577 пользователей, видевших рекламу, и 23 524, видевших на тех же местах социальную рекламу.
2. Реклама дала 7,7 дополнительной конверсии на 1000 пользователей: 25,5 против 17,9, то есть примерно +43 %.
3. Случайностью это объяснить трудно. Реальный эффект, скорее всего, от 6,0 до 9,4 конверсии на 1000 пользователей.
4. Рекомендую продолжать, если реклама окупается даже при нижней оценке: затраты на 1000 пользователей < 6 × прибыль с одной конверсии. При среднем 24,8 показа на пользователя это значит, что прибыль с конверсии должна быть больше примерно 4,2 × цены 1000 показов.
5. Ограничения: в данных нет денег, дат и определения конверсии, заявленный сплит неизвестен, а проверить сопоставимость групп по признакам до показа нельзя.